# AI for Molecules — Hands-on Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Corteswain/AI4molecules-tutorial/blob/main/notebooks/AI4molecules_tutorial.ipynb)

In this 75-minute tutorial you'll build a molecular property predictor from scratch, making three modeling decisions along the way:

1. **Representation** — how do we turn a molecule (a SMILES string) into numbers a model can use?
2. **Dataset splitting** — how do we split data into train/test so our performance estimate is trustworthy?
3. **Model choice** — a classical ML model (Random Forest / XGBoost) on hand-crafted features, or a graph neural network (Chemprop) that learns its own representation?

At each step, a few options are already implemented as functions in `utils/`. **You choose which one to plug in**, run it, and then we compare notes as a group before moving to the next step.

**Task:** predict aqueous solubility (logS) from molecular structure, using the [ESOL dataset](https://pubs.acs.org/doi/10.1021/ci034243x) (1,128 small molecules).

**Agenda (75 min)**

| Time | Section |
|---|---|
| 0–8 min | Setup |
| 8–23 min | Step 1: Representation |
| 23–41 min | Step 2: Splitting |
| 41–66 min | Step 3: Model choice |
| 66–75 min | Wrap-up & comparison |


## Setup

Run the cell below once. On Colab it installs the needed packages and pulls in the `utils/` code and dataset from the tutorial repo (takes ~1-2 minutes).

In [ ]:
import sys
import os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q rdkit scikit-learn xgboost chemprop pandas matplotlib
    if not os.path.exists("AI4molecules-tutorial"):
        !git clone --quiet https://github.com/Corteswain/AI4molecules-tutorial.git
    %cd AI4molecules-tutorial

sys.path.insert(0, ".")
print("Setup done. Running in Colab:", IN_COLAB)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.representations import REPRESENTATIONS
from utils.splitting import SPLITTERS
from utils.models import MODEL_RUNNERS, run_chemprop

TARGET = "measured_log_solubility_mol_per_L"
df = pd.read_csv("data/esol.csv")
print(f"{len(df)} molecules")
df.head()

In [ ]:
df[TARGET].hist(bins=30)
plt.xlabel("measured log solubility (mol/L)")
plt.ylabel("count")
plt.title("ESOL target distribution")
plt.show()

## Step 1 — Representation

How do we turn a molecule into numbers a model can use? Three options are implemented in `utils/representations.py`:

- **`morgan`** — Morgan (circular) fingerprints: a 2048-bit vector encoding which local substructures are present.
- **`maccs`** — MACCS keys: a fixed 166-bit vector, each bit a specific, human-named structural pattern (e.g. "has a ring of size 6").
- **`rdkit_descriptors`** — a handful of global physicochemical descriptors: molecular weight, LogP, TPSA, H-bond donors/acceptors, rotatable bonds, ring counts...

Pick one below.

In [ ]:
# ---- YOUR CHOICE ----
REPRESENTATION = "morgan"  # options: "morgan", "maccs", "rdkit_descriptors"
# ----------------------

featurize = REPRESENTATIONS[REPRESENTATION]
X_all = featurize(df["smiles"].tolist())
y_all = df[TARGET].values
print(f"{REPRESENTATION}: X_all.shape = {X_all.shape}")

**Discuss:**
- How many numbers describe each molecule here? What structural information might this representation lose?
- Would two molecules a chemist considers very similar end up with similar feature vectors?
- Could you look at this feature vector and explain *why* the model made a given prediction?

## Step 2 — Dataset splitting

How we split molecules into train/test changes what our evaluation actually measures. Three options are implemented in `utils/splitting.py`:

- **`random`** — a plain i.i.d. shuffle-and-split. No relationship between molecules is considered.
- **`scaffold`** — molecules are grouped by [Bemis-Murcko scaffold](https://en.wikipedia.org/wiki/Bemis%E2%80%93Murcko_scaffold) (their shared ring/core structure); whole scaffold groups go to either train or test, so near-identical molecules can't leak across the split.
- **`cluster`** — molecules are clustered by fingerprint similarity (KMeans), and whole clusters are held out for test. Similar idea to scaffold splitting, but based on overall structural similarity rather than an exact scaffold match.

Pick one below.

In [ ]:
# ---- YOUR CHOICE ----
SPLIT_METHOD = "random"  # options: "random", "scaffold", "cluster"
# ----------------------

split_fn = SPLITTERS[SPLIT_METHOD]
train_idx, test_idx = split_fn(df, test_size=0.2, seed=42)

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)
X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]

print(f"{SPLIT_METHOD} split: {len(train_idx)} train / {len(test_idx)} test")

In [ ]:
plt.hist(y_train, bins=30, alpha=0.6, label="train", density=True)
plt.hist(y_test, bins=30, alpha=0.6, label="test", density=True)
plt.xlabel("measured log solubility (mol/L)")
plt.ylabel("density")
plt.legend()
plt.title(f"Target distribution: {SPLIT_METHOD} split")
plt.show()

**Discuss:**
- Do train and test look like they come from the same distribution?
- With `scaffold` or `cluster`, are you testing *interpolation* (molecules similar to what the model has seen) or *extrapolation* (genuinely new chemistry)?
- Which split would you trust more as an estimate of performance on molecules made in a lab next year?

## Step 3 — Model choice

Three options are implemented in `utils/models.py`:

- **`random_forest`** / **`xgboost`** — classical ML models trained on the feature vectors (`X_train`/`X_test`) from Step 1.
- **`chemprop`** — a message-passing graph neural network (MPNN) that learns its own representation directly from the molecular graph. It **ignores your Step 1 choice** and instead trains directly on `train_df`/`test_df` (raw SMILES + target).

Pick one below. (Chemprop takes ~20-40 seconds to train.)

In [ ]:
# ---- YOUR CHOICE ----
MODEL = "random_forest"  # options: "random_forest", "xgboost", "chemprop"
# ----------------------

if MODEL == "chemprop":
    # Chemprop learns its own representation from the molecular graph directly —
    # it bypasses the REPRESENTATION you picked in Step 1 and uses train_df/test_df instead.
    result = run_chemprop(train_df, test_df, target_col=TARGET, epochs=30)
else:
    result = MODEL_RUNNERS[MODEL](X_train, y_train, X_test, y_test)

print(f"{MODEL} ({REPRESENTATION} representation / {SPLIT_METHOD} split)")
print(f"RMSE = {result['rmse']:.3f}   R2 = {result['r2']:.3f}")

In [ ]:
y_pred = result["y_pred"]
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot(lims, lims, "k--", linewidth=1)
plt.xlabel("measured")
plt.ylabel("predicted")
plt.title(f"{MODEL} — {REPRESENTATION} / {SPLIT_METHOD} split")
plt.show()

**Discuss:**
- How does your RMSE/R² compare to what your neighbor got with a different choice at any step?
- Random Forest / XGBoost only ever see the Step 1 features — if performance is poor, is that the model's fault or the representation's?
- Chemprop skipped Step 1 entirely. Did it do better or worse than the hand-crafted features here? (Hint: ESOL's target was historically fit using a formula built from LogP, molecular weight, and aromaticity — some of the exact descriptors in `rdkit_descriptors`.)

## Bonus — compare every combination

If time allows, run the two cells below to sweep every (representation × split × model) combination and compare them side by side. The classical combinations run in seconds; adding chemprop across all three splits takes roughly a minute more.

In [ ]:
results = []
for rep_name in REPRESENTATIONS:
    Xa = REPRESENTATIONS[rep_name](df["smiles"].tolist())
    for split_name in SPLITTERS:
        tr_idx, te_idx = SPLITTERS[split_name](df, test_size=0.2, seed=42)
        Xtr, Xte = Xa[tr_idx], Xa[te_idx]
        ytr, yte = y_all[tr_idx], y_all[te_idx]
        for model_name in MODEL_RUNNERS:
            r = MODEL_RUNNERS[model_name](Xtr, ytr, Xte, yte)
            results.append({
                "representation": rep_name, "split": split_name, "model": model_name,
                "rmse": r["rmse"], "r2": r["r2"],
            })

results_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
results_df

In [ ]:
# Optional: also benchmark chemprop across each split (~1 minute total)
for split_name in SPLITTERS:
    tr_idx, te_idx = SPLITTERS[split_name](df, test_size=0.2, seed=42)
    tr_df = df.iloc[tr_idx].reset_index(drop=True)
    te_df = df.iloc[te_idx].reset_index(drop=True)
    r = run_chemprop(tr_df, te_df, target_col=TARGET, epochs=30)
    results.append({
        "representation": "learned (chemprop)", "split": split_name, "model": "chemprop",
        "rmse": r["rmse"], "r2": r["r2"],
    })

results_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
labels = results_df["representation"] + " / " + results_df["split"] + " / " + results_df["model"]
ax.barh(labels, results_df["rmse"])
ax.set_xlabel("RMSE (lower is better)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Wrap-up

Today you made three modeling decisions and saw how each one changes what "good performance" even means:

- **Representation** shapes what information is available to the model at all.
- **Splitting** shapes what question your evaluation is actually answering — interpolation vs. extrapolation.
- **Model choice** interacts with both: some models need you to hand-design good features (Random Forest, XGBoost); others learn a representation end-to-end (Chemprop), trading interpretability and speed for flexibility.

**Keep exploring:**
- Try other [MoleculeNet](https://moleculenet.org/) datasets (BBBP, Tox21, FreeSolv) with the same functions.
- Add hyperparameter tuning (`GridSearchCV`, or Chemprop's `--depth` / `--hidden-size` / `--epochs`).
- Open `utils/representations.py`, `utils/splitting.py`, `utils/models.py` to see how each function is implemented, and try writing your own.

Repo: https://github.com/Corteswain/AI4molecules-tutorial